In [113]:
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
import numpy as np

In [114]:
RAW_XML_PATH = Path("../data/export.xml")
PROCESSED_PATH = Path("../data/processed.csv")

In [115]:
# --- 1. Parse XML
tree = ET.parse(RAW_XML_PATH)
root = tree.getroot()

attrs = set()
for i, record in enumerate(root.findall('Record')):
    attrs.update(record.attrib.keys())
    if i > 1000:
        break
print(attrs)


{'endDate', 'type', 'sourceName', 'value', 'creationDate', 'startDate', 'sourceVersion', 'unit'}


In [116]:
root

<Element 'HealthData' at 0x41a6c6ac0>

In [117]:
records = []
for record in root.findall('Record'):
    rtype = record.get('type')
    start = record.get('startDate')
    end = record.get('endDate')
    value = record.get('value')
    unit = record.get('unit')
    source = record.get('sourceName')
    records.append((rtype, start, end, value, unit, source))


In [118]:
dupes = df[df.duplicated(subset=['start', 'end','rtype', 'value'], keep=False)]
dupes

,rtype,start,end,value,unit,source,window
934048,StepCount,2026-04-01 07:50:03-04:00,2026-04-01 07:50:34-04:00,29.000000,count,Kay’s Apple Watch,2
934052,StepCount,2026-04-01 07:50:03-04:00,2026-04-01 07:50:34-04:00,29.000000,count,Kay’s Apple Watch,2
934155,StepCount,2026-04-02 13:07:51-04:00,2026-04-02 13:08:09-04:00,18.000000,count,Kay’s Apple Watch,2
934156,StepCount,2026-04-02 13:07:51-04:00,2026-04-02 13:08:09-04:00,18.000000,count,Kay’s Apple Watch,2
934494,StepCount,2026-04-06 13:58:35-04:00,2026-04-06 14:01:37-04:00,217.000000,count,Kay’s Apple Watch,2
934495,StepCount,2026-04-06 13:58:35-04:00,2026-04-06 14:01:37-04:00,217.000000,count,Kay’s Apple Watch,2
934580,StepCount,2026-04-07 11:41:12-04:00,2026-04-07 11:43:49-04:00,119.000000,count,Kay’s Apple Watch,2
934581,StepCount,2026-04-07 11:41:12-04:00,2026-04-07 11:43:49-04:00,119.000000,count,Kay’s Apple Watch,2
2191827,DietaryFatTotal,2025-08-07 15:15:37-04:00,2025-08-07 15:15:37-04:00,0.390000,g,Bevel,1
2191830,DietaryFatTotal,2025-08-07 15:15:37-04:00,2025-08-07 15:15:37-04:00,0.390000,g,Bevel,1


In [119]:
df_all = pd.DataFrame(records, columns=["rtype","start","end","value","unit","source"])
df_all['rtype'] = df_all['rtype'].str.split("Identifier").str[-1]
df_all['rtype'].unique()

array(['BloodGlucose', 'DietaryWater', 'BodyMassIndex', 'Height',
       'BodyMass', 'HeartRate', 'OxygenSaturation', 'RespiratoryRate',
       'BodyFatPercentage', 'LeanBodyMass', 'StepCount',
       'DistanceWalkingRunning', 'BasalEnergyBurned',
       'ActiveEnergyBurned', 'FlightsClimbed', 'DietaryFatTotal',
       'DietaryFatPolyunsaturated', 'DietaryFatMonounsaturated',
       'DietaryFatSaturated', 'DietaryCholesterol', 'DietarySodium',
       'DietaryCarbohydrates', 'DietaryFiber', 'DietarySugar',
       'DietaryEnergyConsumed', 'DietaryProtein', 'DietaryVitaminA',
       'DietaryVitaminB6', 'DietaryVitaminB12', 'DietaryVitaminC',
       'DietaryVitaminD', 'DietaryVitaminE', 'DietaryVitaminK',
       'DietaryCalcium', 'DietaryIron', 'DietaryThiamin',
       'DietaryRiboflavin', 'DietaryNiacin', 'DietaryFolate',
       'DietaryBiotin', 'DietaryPantothenicAcid', 'DietaryPhosphorus',
       'DietaryIodine', 'DietaryMagnesium', 'DietaryZinc',
       'DietarySelenium', 'DietaryCoppe

In [120]:
df_all['source'].unique()

array(['Lingo', 'Bevel', 'Health', 'VeSync', 'Kay’s Apple\xa0Watch',
       'OKK', '薄荷健康', 'MyFitnessPal', 'Cal AI', 'Kay’s iPhone',
       'Blood Oxygen', 'Kay14', 'iPhone', 'GymKit', 'Nike Run Club',
       'StepsApp', 'Sleep', 'Sleep Cycle', 'Cycle Tracking', '番茄ToDo'],
      dtype=object)

In [121]:
df_all[df_all['source']=='Sleep Cycle']

,rtype,start,end,value,unit,source
3184255,SleepAnalysis,2020-06-09 22:37:33 -0400,2020-06-10 05:47:05 -0400,HKCategoryValueSleepAnalysisInBed,None,Sleep Cycle
3184256,SleepAnalysis,2020-06-09 22:56:10 -0400,2020-06-10 05:47:05 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3184257,SleepAnalysis,2020-06-10 23:32:49 -0400,2020-06-11 03:32:09 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3184258,SleepAnalysis,2020-06-11 03:52:26 -0400,2020-06-11 05:56:49 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3184259,SleepAnalysis,2020-06-10 23:24:42 -0400,2020-06-11 06:10:21 -0400,HKCategoryValueSleepAnalysisInBed,None,Sleep Cycle
...,...,...,...,...,...,...
3188454,SleepAnalysis,2023-10-23 22:07:11 -0400,2023-10-24 06:13:41 -0400,HKCategoryValueSleepAnalysisInBed,None,Sleep Cycle
3188692,SleepAnalysis,2023-11-01 22:40:44 -0400,2023-11-02 02:35:02 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3188693,SleepAnalysis,2023-11-02 03:07:52 -0400,2023-11-02 04:32:56 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3188694,SleepAnalysis,2023-11-02 05:02:47 -0400,2023-11-02 05:49:03 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle


In [122]:
df_all[df_all['rtype']=='SleepAnalysis']['source'].unique()

array(['Sleep Cycle', 'Kay’s Apple\xa0Watch', 'Kay14'], dtype=object)

In [123]:
df_all['start'] = pd.to_datetime(df_all['start'], errors='coerce')
df_all['end'] = pd.to_datetime(df_all['end'], errors='coerce')

In [124]:
w1_start, w1_end = pd.Timestamp("2025-07-28", tz="UTC-04:00"), pd.Timestamp("2025-07-28", tz="UTC-04:00") + pd.Timedelta(days=14)
w2_start, w2_end = pd.Timestamp("2026-03-25", tz="UTC-04:00"), pd.Timestamp("2026-03-25", tz="UTC-04:00") + pd.Timedelta(days=14)
df = df_all[
    (((df_all['start'] >= w1_start) & (df_all['start'] <= w1_end)) |
     ((df_all['start'] >= w2_start) & (df_all['start'] <= w2_end)))
].copy()

df['window'] = np.where(df['start'] <= w1_end, 1, 2)

In [125]:
sources_to_keep = ['Lingo', 'Bevel', 'Kay\u2019s Apple\xa0Watch']
variables_to_keep = ['BloodGlucose', 'HeartRate','StepCount','ActiveEnergyBurned',
           'DietaryCarbohydrates','DietaryProtein','DietaryFatTotal',
           'DietaryFiber','DietaryEnergyConsumed','SleepAnalysis']
df = df[df['rtype'].isin(variables_to_keep) & df['source'].isin(sources_to_keep)]

In [126]:
df.columns

Index(['rtype', 'start', 'end', 'value', 'unit', 'source', 'window'], dtype='object')

In [127]:
df_sorted = df[df['rtype']=='DietaryEnergyConsumed'].sort_values('start').reset_index(drop=True)
df_sorted['value'] = df_sorted['value'].astype(float)

candidates = []
for i in range(len(df_sorted)-1):
    row_a = df_sorted.iloc[i]
    row_b = df_sorted.iloc[i+1]
    time_diff = (row_b['start'] - row_a['start'])
    if row_a['value'] == 0:
        continue
    value_diff_pct = abs(row_a['value'] - row_b['value']) / row_a['value']
    if time_diff < pd.Timedelta(minutes=10) and value_diff_pct < 0.10:
        candidates.append((row_a, row_b, time_diff, value_diff_pct))

for a, b, td, vd in candidates:
    print(f"{a['start']} ({a['value']}) vs {b['start']} ({b['value']}) — Δt={td}, Δval={vd:.1%}")

2025-08-05 08:50:50-04:00 (410.828) vs 2025-08-05 08:54:19-04:00 (421.413) — Δt=0 days 00:03:29, Δval=2.6%
2025-08-05 13:01:49-04:00 (30.0) vs 2025-08-05 13:02:58-04:00 (32.5) — Δt=0 days 00:01:09, Δval=8.3%
2025-08-07 23:50:48-04:00 (45.6) vs 2025-08-07 23:50:48-04:00 (45.6) — Δt=0 days 00:00:00, Δval=0.0%
2025-08-08 14:14:39-04:00 (164.962) vs 2025-08-08 14:14:39-04:00 (179.656) — Δt=0 days 00:00:00, Δval=8.9%
2025-08-10 14:18:08-04:00 (100.3) vs 2025-08-10 14:18:08-04:00 (100.0) — Δt=0 days 00:00:00, Δval=0.3%
2026-03-29 08:38:08-04:00 (100.0) vs 2026-03-29 08:38:08-04:00 (93.21) — Δt=0 days 00:00:00, Δval=6.8%
2026-03-30 08:56:51-04:00 (180.165) vs 2026-03-30 08:56:51-04:00 (196.94) — Δt=0 days 00:00:00, Δval=9.3%
2026-04-01 14:22:45-04:00 (95.0) vs 2026-04-01 14:22:45-04:00 (91.875) — Δt=0 days 00:00:00, Δval=3.3%


In [128]:
dupe_indices_to_drop = []
for a, b, td, vd in candidates:
    dupe_indices_to_drop.append(b.name)  # drop the second occurrence, keep the first

df = df.drop(index=dupe_indices_to_drop)

In [129]:
df_sleep = df[df['rtype'] == 'SleepAnalysis'].copy()
df = df[df['rtype'] != 'SleepAnalysis'].copy()

df['start'] = pd.to_datetime(df['start'], errors='coerce')
df['end'] = pd.to_datetime(df['end'], errors='coerce')
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['value'])

df_sleep['start'] = pd.to_datetime(df_sleep['start'], errors='coerce')
df_sleep['end'] = pd.to_datetime(df_sleep['end'], errors='coerce')
# value stays as string category — don't coerce to numeric

In [160]:
food = df[df['rtype'] == 'DietaryEnergyConsumed'].sort_values('start').copy()
gap = food['start'].diff().dt.total_seconds().div(60).fillna(999)
food['meal_id'] = (gap > 15).cumsum()

meals = food.groupby('meal_id').agg(
    meal_start=('start', 'min'),
    meal_end=('end', 'max'),
    total_energy=('value', 'sum'),
    n_entries=('value', 'count'),
    window=('window', 'first')
).reset_index()

macros = ['DietaryEnergyConsumed', 'DietaryProtein', 'DietaryCarbohydrates', 'DietaryFatTotal', 'DietaryFiber']
wide = df[df['rtype'].isin(macros)].pivot_table(index='start', columns='rtype', values='value', aggfunc='sum').reset_index()
wide = pd.merge_asof(wide.sort_values('start'), food[['start', 'meal_id']].sort_values('start'), on='start', direction='nearest', tolerance=pd.Timedelta('1min'))

# guard: merge_asof with a 1min tolerance leaves meal_id NaN for any macro record
# not near a meal; groupby('meal_id') then silently drops those. Surface it instead.
_unmatched = int(wide['meal_id'].isna().sum())
if _unmatched:
    print(f"warning: {_unmatched} macro record(s) >1min from any meal — dropped from meal totals")

meal_dataset = wide.groupby('meal_id').sum(numeric_only=True).reset_index()
meal_dataset = meal_dataset.merge(meals[['meal_id', 'meal_start', 'meal_end', 'window']], on='meal_id', how='left')

In [161]:
glucose = df[df['rtype'] == 'BloodGlucose'][['start', 'value']].sort_values('start').rename(columns={'value': 'glucose'})

def glucose_features(meal_start, glucose_df, baseline_min=15, response_hr=2):
    baseline_window = glucose_df[(glucose_df['start'] >= meal_start - pd.Timedelta(minutes=baseline_min)) & (glucose_df['start'] < meal_start)]
    response_window = glucose_df[(glucose_df['start'] >= meal_start) & (glucose_df['start'] <= meal_start + pd.Timedelta(hours=response_hr))]

    if baseline_window.empty or response_window.empty:
        return pd.Series({'baseline_glucose': np.nan, 'peak_rise': np.nan, 'iauc': np.nan,
                           'n_baseline_pts': len(baseline_window), 'n_response_pts': len(response_window)})

    baseline = baseline_window['glucose'].mean()
    peak_rise = response_window['glucose'].max() - baseline

    t = (response_window['start'] - meal_start).dt.total_seconds() / 60
    g_clipped = (response_window['glucose'] - baseline).clip(lower=0)
    iauc = np.trapezoid(g_clipped, t)

    return pd.Series({'baseline_glucose': baseline, 'peak_rise': peak_rise, 'iauc': iauc,
                       'n_baseline_pts': len(baseline_window), 'n_response_pts': len(response_window)})

feats = meal_dataset['meal_start'].apply(lambda t: glucose_features(t, glucose))
meal_dataset = pd.concat([meal_dataset, feats], axis=1)

In [162]:
print(meal_dataset[['n_baseline_pts', 'n_response_pts']].describe())
print((meal_dataset['n_baseline_pts'] == 0).sum(), (meal_dataset['n_response_pts'] == 0).sum())

       n_baseline_pts  n_response_pts
count       91.000000       91.000000
mean         2.967033       23.956044
std          0.314485        0.419314
min          0.000000       20.000000
25%          3.000000       24.000000
50%          3.000000       24.000000
75%          3.000000       24.000000
max          3.000000       24.000000
1 0


In [163]:
meal_dataset = meal_dataset[
    (meal_dataset['n_baseline_pts'] > 0) & (meal_dataset['DietaryEnergyConsumed'] < 700)
].copy()


In [164]:
print(meal_dataset['window'].value_counts())

window
2    46
1    41
Name: count, dtype: int64


In [165]:
activity = df[df['rtype'].isin(['StepCount', 'HeartRate', 'ActiveEnergyBurned'])][['rtype', 'start', 'end', 'value']].copy()

def activity_features(meal_start, activity_df, pre_min=60, post_min=60):
    pre = activity_df[(activity_df['start'] >= meal_start - pd.Timedelta(minutes=pre_min)) & (activity_df['start'] < meal_start)]
    post = activity_df[(activity_df['start'] > meal_start) & (activity_df['start'] <= meal_start + pd.Timedelta(minutes=post_min))]

    def agg(window):
        steps = window[window['rtype'] == 'StepCount']['value'].sum()
        active_energy = window[window['rtype'] == 'ActiveEnergyBurned']['value'].sum()
        hr_mean = window[window['rtype'] == 'HeartRate']['value'].mean()
        return steps, active_energy, hr_mean

    pre_steps, pre_energy, pre_hr = agg(pre)
    post_steps, post_energy, post_hr = agg(post)

    return pd.Series({
        'pre_steps': pre_steps, 'pre_active_energy': pre_energy, 'pre_hr': pre_hr,
        'post_steps': post_steps, 'post_active_energy': post_energy, 'post_hr': post_hr,
        'walked_after_eating': int(post_steps > 200)  # threshold worth eyeballing against your actual data distribution
    })

act_feats = meal_dataset['meal_start'].apply(lambda t: activity_features(t, activity))
meal_dataset = pd.concat([meal_dataset, act_feats], axis=1)

In [166]:
sleep = df[df['rtype'] == 'SleepAnalysis'][['start', 'end', 'value']].copy()
asleep_categories = [v for v in sleep['value'].unique() if 'Asleep' in v]  # excludes Awake/InBed

def sleep_hours_before(meal_start, sleep_df, lookback_hr=18):
    window = sleep_df[(sleep_df['end'] <= meal_start) & (sleep_df['end'] >= meal_start - pd.Timedelta(hours=lookback_hr)) & (sleep_df['value'].isin(asleep_categories))]
    total_sec = (window['end'] - window['start']).dt.total_seconds().sum()
    return total_sec / 3600

meal_dataset['hours_slept'] = meal_dataset['meal_start'].apply(lambda t: sleep_hours_before(t, sleep))

In [167]:
export = meal_dataset[['meal_id', 'meal_start', 'meal_end', 'DietaryCarbohydrates', 'DietaryEnergyConsumed']].copy()
export['meal_start'] = export['meal_start'].dt.tz_localize(None)
export['meal_end'] = export['meal_end'].dt.tz_localize(None)
export = export 
export['carb_source'] = ''
export['GI'] = ''
# export.to_excel('../data/add_carb_source.xlsx', index=False)

In [168]:
carb_source_export = pd.read_excel('../data/add_carb_source_edited.xlsx')

distinct_sources = sorted(carb_source_export['carb_source'].dropna().unique())
print(distinct_sources)
print(len(distinct_sources))

['almonds', 'apple', 'bagel', 'banana', 'berries', 'biscotti', 'blueberries', 'bread', 'cheddar', 'chia seeds', 'chicken', 'crispbread', 'dark chocolate', 'dates', 'egg', 'farro', 'fried chicken', 'honey', 'jam', 'kefir', 'kiwi', 'kombucha', 'lentil', 'milk', 'mixed non-starchy vegetables', 'nuts', 'oats', 'pasta', 'peanuts', 'pear', 'pinapple', 'plum', 'potato', 'pretzels', 'protein bar', 'puff pastry', 'rice', 'rice noodle', 'rye bread', 'samosa', 'sauce', 'sticky rice', 'watermelon', 'yogurt']
44


In [ ]:
# Overlay labels from the auto-label + human-review loop (src/carb_source_labeler.py:
# photo -> vision model proposal -> reviewed/corrected in review_carb_labels.xlsx ->
# merged by --apply-review). final_carb_labels.csv is the production input and wins
# wherever a meal went through review; meals never reviewed (e.g. no photo) fall back
# to the hand label. add_carb_source_edited.xlsx stays frozen as the original
# hand-labeled ground truth that the labeler's eval is scored against.
final_labels = pd.read_csv('../data/final_carb_labels.csv').rename(columns={'carb_source': 'carb_source_reviewed'})
carb_source_export = carb_source_export.merge(final_labels, on='meal_id', how='left')
carb_source_export['carb_source'] = carb_source_export['carb_source_reviewed'].fillna(carb_source_export['carb_source'])
carb_source_export = carb_source_export.drop(columns=['carb_source_reviewed'])
print(f"reviewed labels applied to {len(final_labels)} meals; "
      f"{carb_source_export['carb_source'].notna().sum()} meals labeled total")

In [169]:
gi_lookup = {
    'almonds': 0,
    'apple': 36,
    'bagel': 69,
    'banana': 51,
    'berries': 25,
    'biscotti': 60,
    'blueberries': 53,
    'bread': 75,
    'cheddar': 0,
    'chia seeds': 1,
    'chicken': 0,
    'crispbread': 65,
    'dark chocolate': 23,
    'dates': 42,
    'egg': 0,
    'farro': 45,
    'fried chicken': 0,
    'honey': 61,
    'jam': 65,
    'kefir': 32,
    'kiwi': 50,
    'kombucha': 40,
    'lentil': 32,
    'milk': 30,
    'mixed non-starchy vegetables': 15,
    'nuts': 15,
    'oats': 55,
    'pasta': 50,
    'peanuts': 13,
    'pear': 38,
    'pinapple': 59,
    'plum': 40,
    'potato': 85,
    'pretzels': 83,
    'protein bar': 35,        # rough estimate — these vary widely; check label if you want more precision
    'puff pastry': 59,
    'rice': 73,
    'rice noodle': 53,
    'rye bread': 58,
    'samosa': 60,             # fried, starch-wrapped — moderate-high GI estimate
    'sauce': 36,            # korma
    'sticky rice': 98,
    'watermelon': 76,
    'yogurt': 41,
}

In [170]:
carb_source_export['carb_source'] = carb_source_export['carb_source'].str.strip().str.lower()
carb_source_export['GI'] = carb_source_export['carb_source'].map(gi_lookup)

missing = carb_source_export[carb_source_export['GI'].isna()]
print(missing[['meal_id', 'carb_source']])

Empty DataFrame
Columns: [meal_id, carb_source]
Index: []


In [171]:
meal_dataset = meal_dataset.merge(carb_source_export[['meal_id', 'carb_source', 'GI']], on='meal_id', how='left')
assert meal_dataset['GI'].isna().sum() == 0, "some meals missing GI after merge"

meal_dataset['glycemic_load'] = (meal_dataset['GI'] * meal_dataset['DietaryCarbohydrates']) / 100

In [172]:
meal_dataset.to_parquet('../data/meal_dataset.parquet')
glucose.to_parquet('../data/glucose_timeseries.parquet')